# 03. Structural metrics: Exact Match and Component F1

These compare the generated query *string* to the reference (gold) query *string*, no database required. They are the cheapest signals to compute and the easiest to interpret, but they share one well-known weakness: two queries can be semantically identical and lexically different (different variable names, different MATCH order, different but equivalent WHERE clauses).

We mitigate this with **canonicalisation**, a normalisation pass applied to both sides before comparison:

1. Strip line and block comments.
2. Alpha-rename pattern bindings (`s`, `x`, `liker`, …) to a positional sequence (`v0`, `v1`, …).
3. Uppercase reserved keywords; leave identifiers and literal strings alone.

After canonicalisation, two queries that differ only in whitespace, capitalisation, or variable choice will compare equal.

Two metrics are reported:

- **Exact Match (EM)**: binary equality of the canonical strings.
- **Component F1**: decompose each query into clause components (node labels, edge types, directions, WHERE/RETURN/ORDER tokens, LIMIT, aggregations), compute F1 per component, then macro-average. This is the diagnostic metric: when EM = 0, the per-component breakdown tells you *where* the LLM is failing.

Before we report numbers on real records, we run an **identity sanity test**: feed each gold query as its own "translation" and assert EM = True, F1 = 1.0. Any failure means our canonicalisation is buggy and downstream numbers are not trustworthy.

In [ ]:
from __future__ import annotations

import json
import re
from collections import Counter
from dataclasses import dataclass, field
from pathlib import Path
from typing import Literal

import pandas as pd
import yaml

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "pyproject.toml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent

OUTPUTS_DIR = REPO_ROOT / "evaluation" / "outputs"
RECORDS_PATH = OUTPUTS_DIR / "records.json"
DATASETS_DIR = REPO_ROOT / "evaluation" / "datasets"
OUT_CSV = OUTPUTS_DIR / "metrics_structural.csv"

Dialect = Literal["cypher", "aql"]

## Tokeniser

Splits a query string into tokens while preserving (a) quoted literals as single units (so `'Supplier#000000666'` is one token), (b) numeric literals (so `1995` is not split at the digit boundary), (c) two-character punctuation like `->` and `<-`. Everything else falls through to single-character punctuation.

In [ ]:
_STRING_LITERAL = re.compile(r"'(?:[^'\\]|\\.)*'|\"(?:[^\"\\]|\\.)*\"")
_NUMBER_LITERAL = re.compile(r"\b\d+(?:\.\d+)?\b")
_PUNCT = re.compile(r"[(){}\[\],;]|->|<-|<>|>=|<=|==|!=|=|<|>|\.|:|\+|\-|\*|/|\|")
_IDENT = re.compile(r"[A-Za-z_][A-Za-z0-9_]*")
_COMMENT_LINE = re.compile(r"//[^\n]*|--[^\n]*")
_COMMENT_BLOCK = re.compile(r"/\*.*?\*/", re.DOTALL)


def strip_comments(query: str) -> str:
    query = _COMMENT_BLOCK.sub(" ", query)
    return _COMMENT_LINE.sub(" ", query)


def tokenize(query: str) -> list[str]:
    text = strip_comments(query)
    tokens: list[str] = []
    i, n = 0, len(text)
    while i < n:
        if text[i].isspace():
            i += 1
            continue
        for pat in (_STRING_LITERAL, _NUMBER_LITERAL, _IDENT, _PUNCT):
            m = pat.match(text, i)
            if m:
                tokens.append(m.group(0))
                i = m.end()
                break
        else:
            tokens.append(text[i])
            i += 1
    return tokens


tokenize("MATCH (s:Supplier {name: 'X'})-[:HAS]->(x) RETURN s.name LIMIT 10")

## Keyword sets and clause heads

The reserved-word list for each dialect. We uppercase only these during canonicalisation, leaving identifiers (label names, property names) untouched.

In [ ]:
_CYPHER_KEYWORDS = frozenset([
    "MATCH", "OPTIONAL", "WHERE", "RETURN", "WITH", "ORDER", "BY", "LIMIT", "SKIP",
    "UNWIND", "CALL", "CREATE", "MERGE", "DELETE", "DETACH", "SET", "REMOVE", "UNION",
    "DISTINCT", "AS", "AND", "OR", "NOT", "IN", "IS", "NULL", "TRUE", "FALSE",
    "ASC", "DESC", "CONTAINS", "STARTS", "ENDS", "WHEN", "CASE", "THEN", "ELSE", "END", "FOREACH",
])
_AQL_KEYWORDS = frozenset([
    "FOR", "IN", "LET", "RETURN", "FILTER", "SORT", "LIMIT", "COLLECT",
    "OUTBOUND", "INBOUND", "ANY", "GRAPH", "AGGREGATE", "WITH", "INTO", "DISTINCT",
    "ASC", "DESC", "AND", "OR", "NOT", "TRUE", "FALSE", "NULL",
    "INSERT", "UPDATE", "REPLACE", "REMOVE", "UPSERT", "OPTIONS", "LIKE",
])
_CYPHER_CLAUSE_HEADS = (
    "MATCH", "OPTIONAL MATCH", "WHERE", "RETURN", "WITH", "ORDER BY", "LIMIT", "SKIP",
    "UNWIND", "CALL", "CREATE", "MERGE", "DELETE", "DETACH DELETE", "SET", "REMOVE", "UNION",
)
_AQL_CLAUSE_HEADS = (
    "FOR", "LET", "FILTER", "SORT", "LIMIT", "COLLECT", "RETURN",
    "INSERT", "UPDATE", "REPLACE", "REMOVE", "UPSERT",
)


def keywords_for(dialect: Dialect) -> frozenset[str]:
    return _CYPHER_KEYWORDS if dialect == "cypher" else _AQL_KEYWORDS


def clause_heads_for(dialect: Dialect) -> tuple[str, ...]:
    return _CYPHER_CLAUSE_HEADS if dialect == "cypher" else _AQL_CLAUSE_HEADS

## Canonicalisation

Pattern variables in Cypher are introduced inside `(` or `[`: `(s:Supplier ...)`, `[k:KNOWS ...]`. AQL introduces them via `FOR var IN ...` and `LET var = ...`. We collect every binding in source order and rename them to `v0`, `v1`, … so the canonical form is stable regardless of what name the LLM picked.

In [ ]:
_CYPHER_BINDING = re.compile(r"[(\[]([A-Za-z_][A-Za-z0-9_]*)\b")
_AQL_BINDING = re.compile(r"\b(?:FOR|LET)\s+([A-Za-z_][A-Za-z0-9_]*)\b", re.IGNORECASE)


def alpha_rename(query: str, bindings: list[str]) -> str:
    rename: dict[str, str] = {}
    for name in bindings:
        if name not in rename:
            rename[name] = f"v{len(rename)}"
    if not rename:
        return query
    pattern = re.compile(
        r"\b(" + "|".join(re.escape(n) for n in sorted(rename, key=len, reverse=True)) + r")\b"
    )
    return pattern.sub(lambda m: rename[m.group(1)], query)


def canonicalize(query: str, dialect: Dialect) -> str:
    text = strip_comments(query)
    bindings_re = _CYPHER_BINDING if dialect == "cypher" else _AQL_BINDING
    bindings = [m.group(1) for m in bindings_re.finditer(text)]
    text = alpha_rename(text, bindings)
    kws = keywords_for(dialect)
    out: list[str] = []
    for t in tokenize(text):
        if t.startswith("'") or t.startswith('"'):
            out.append(t)
        else:
            up = t.upper()
            out.append(up if up in kws else t)
    return " ".join(out)


def exact_match(translated: str, expected: str, dialect: Dialect) -> bool:
    return canonicalize(translated, dialect) == canonicalize(expected, dialect)


canonicalize("MATCH (s:Supplier {name:'X'}) return s.name", "cypher")

## Component extraction

We decompose each query into eight buckets and compute F1 per bucket, then average.

**Cypher**:
- `node_labels`: what comes after `:` inside `(...)` patterns, e.g. `Person`, `Supplier`.
- `edge_types`: what comes after `:` inside `[...]` patterns, e.g. `KNOWS`, `HAS_TAG`.
- `directions`: list of `->` and `<-` occurrences (ordered, with multiplicity).
- `where_tokens`, `return_tokens`, `order_tokens`: token sets from each clause body.
- `limit_value`: first token after `LIMIT`.
- `aggregations`: names of aggregate functions called (`count`, `sum`, `max`, …).

**AQL**:
- `node_labels`: collection names in `FOR var IN <Collection>`.
- `edge_types`: collection names following `OUTBOUND`/`INBOUND`/`ANY` (the edge collection).
- `directions`: list of `OUTBOUND`/`INBOUND`/`ANY` occurrences.
- `where_tokens` = `FILTER` body, `order_tokens` = `SORT` body.
- `aggregations`: `COUNT`, `SUM`, `MAX`, `AVERAGE`/`AVG`, `LENGTH`, etc.

In [ ]:
_CYPHER_NODE_LABEL = re.compile(r":\s*([A-Z][A-Za-z0-9_]*)")
_CYPHER_EDGE_TYPE = re.compile(r"\[\s*[A-Za-z_]?\w*\s*:\s*([A-Z_][A-Z0-9_]*)\s*[\]\s{]|\[:([A-Z_][A-Z0-9_]*)")
_CYPHER_DIRECTION = re.compile(r"->|<-")
_CYPHER_AGG = re.compile(r"\b(count|sum|min|max|avg|collect)\s*\(", re.IGNORECASE)

_AQL_COLLECTION = re.compile(r"\bFOR\s+[A-Za-z_]\w*\s+IN\s+([A-Z][A-Za-z0-9_]*)", re.IGNORECASE)
_AQL_EDGE_COLLECTION = re.compile(
    r"\b(?:OUTBOUND|INBOUND|ANY)\b\s+(?:[A-Za-z_]\w*\.[A-Za-z_]\w*|[A-Za-z_]\w*)\s+([A-Z][A-Za-z0-9_]*)",
    re.IGNORECASE,
)
_AQL_DIRECTION = re.compile(r"\b(OUTBOUND|INBOUND|ANY)\b", re.IGNORECASE)
_AQL_AGG = re.compile(r"\b(COUNT|SUM|MIN|MAX|AVERAGE|AVG|LENGTH|COLLECT|AGGREGATE)\s*\(", re.IGNORECASE)


@dataclass
class Components:
    node_labels: set[str] = field(default_factory=set)
    edge_types: set[str] = field(default_factory=set)
    directions: list[str] = field(default_factory=list)
    where_tokens: set[str] = field(default_factory=set)
    return_tokens: set[str] = field(default_factory=set)
    order_tokens: set[str] = field(default_factory=set)
    limit_value: str | None = None
    aggregations: set[str] = field(default_factory=set)


def _clause_body(tokens: list[str], heads: tuple[str, ...], labels: tuple[str, ...]) -> list[str]:
    """Return the flat token list from any clause whose head matches `labels`."""
    body: list[str] = []
    i = 0
    in_target = False
    while i < len(tokens):
        matched_head: str | None = None
        consumed = 0
        for h in sorted(heads, key=lambda x: -len(x.split())):
            parts = h.split()
            if i + len(parts) <= len(tokens) and all(tokens[i + k].upper() == parts[k] for k in range(len(parts))):
                matched_head = h
                consumed = len(parts)
                break
        if matched_head is not None:
            in_target = matched_head in labels
            i += consumed
            continue
        if in_target:
            body.append(tokens[i])
        i += 1
    return body


def _components_cypher(query: str) -> Components:
    text = strip_comments(query)
    c = Components()
    c.node_labels = {m.group(1) for m in _CYPHER_NODE_LABEL.finditer(text)}
    for m in _CYPHER_EDGE_TYPE.finditer(text):
        edge = m.group(1) or m.group(2)
        if edge:
            c.edge_types.add(edge)
    # Strip node-label hits from edge_types: node labels are TitleCase while
    # edge types are SCREAMING_SNAKE; intersect them out to avoid false hits.
    c.edge_types -= c.node_labels
    c.directions = _CYPHER_DIRECTION.findall(text)
    c.aggregations = {m.group(1).lower() for m in _CYPHER_AGG.finditer(text)}
    canon = canonicalize(query, "cypher")
    tokens = canon.split(" ")
    c.where_tokens = set(_clause_body(tokens, _CYPHER_CLAUSE_HEADS, ("WHERE",)))
    c.return_tokens = set(_clause_body(tokens, _CYPHER_CLAUSE_HEADS, ("RETURN",)))
    c.order_tokens = set(_clause_body(tokens, _CYPHER_CLAUSE_HEADS, ("ORDER BY",)))
    limit_body = _clause_body(tokens, _CYPHER_CLAUSE_HEADS, ("LIMIT",))
    c.limit_value = limit_body[0] if limit_body else None
    return c


def _components_aql(query: str) -> Components:
    text = strip_comments(query)
    c = Components()
    c.node_labels = {m.group(1) for m in _AQL_COLLECTION.finditer(text)}
    c.edge_types = {m.group(1) for m in _AQL_EDGE_COLLECTION.finditer(text)}
    c.directions = [d.upper() for d in _AQL_DIRECTION.findall(text)]
    c.aggregations = {m.group(1).lower() for m in _AQL_AGG.finditer(text)}
    canon = canonicalize(query, "aql")
    tokens = canon.split(" ")
    c.where_tokens = set(_clause_body(tokens, _AQL_CLAUSE_HEADS, ("FILTER",)))
    c.return_tokens = set(_clause_body(tokens, _AQL_CLAUSE_HEADS, ("RETURN",)))
    c.order_tokens = set(_clause_body(tokens, _AQL_CLAUSE_HEADS, ("SORT",)))
    limit_body = _clause_body(tokens, _AQL_CLAUSE_HEADS, ("LIMIT",))
    c.limit_value = limit_body[0] if limit_body else None
    return c


def components_of(query: str, dialect: Dialect) -> Components:
    return _components_cypher(query) if dialect == "cypher" else _components_aql(query)

## Set / multiset F1

`_set_f1` for set-valued components (labels, edge types, return tokens), `_list_f1` for multisets that respect multiplicity (directions, which we count `->` vs `<-` occurrences). Both special-case the both-empty case as F1 = 1.0 (vacuously equal) so an absent ORDER BY in both sides scores as a match.

In [ ]:
def _set_f1(a: set[str], b: set[str]) -> float:
    if not a and not b:
        return 1.0
    if not a or not b:
        return 0.0
    tp = len(a & b)
    if tp == 0:
        return 0.0
    p, r = tp / len(a), tp / len(b)
    return 2 * p * r / (p + r)


def _list_f1(a: list[str], b: list[str]) -> float:
    if not a and not b:
        return 1.0
    if not a or not b:
        return 0.0
    ac, bc = Counter(a), Counter(b)
    overlap = sum((ac & bc).values())
    if overlap == 0:
        return 0.0
    p, r = overlap / sum(ac.values()), overlap / sum(bc.values())
    return 2 * p * r / (p + r)


def component_f1(translated: str, expected: str, dialect: Dialect) -> dict[str, float]:
    t = components_of(translated, dialect)
    e = components_of(expected, dialect)
    scores = {
        "node_labels": _set_f1(t.node_labels, e.node_labels),
        "edge_types": _set_f1(t.edge_types, e.edge_types),
        "directions": _list_f1(t.directions, e.directions),
        "where": _set_f1(t.where_tokens, e.where_tokens),
        "return": _set_f1(t.return_tokens, e.return_tokens),
        "order": _set_f1(t.order_tokens, e.order_tokens),
        "limit": 1.0 if t.limit_value == e.limit_value else 0.0,
        "aggregations": _set_f1(t.aggregations, e.aggregations),
    }
    scores["overall"] = sum(scores.values()) / len(scores)
    return scores

## Identity sanity test

Feed each gold query as its own "translation". If our metric implementation is sane, both numbers should be 1.0 (or True) on every record. Any failure here is a bug in canonicalisation: fix before trusting the real numbers.

In [ ]:
identity_rows = []
for ds_path in sorted(DATASETS_DIR.glob("*.yaml")):
    data = yaml.safe_load(ds_path.read_text())
    for q in data["queries"]:
        for dialect, key in (("cypher", "expected_cypher"), ("aql", "expected_aql")):
            ref = q[key]
            em = exact_match(ref, ref, dialect)
            cf = component_f1(ref, ref, dialect)["overall"]
            identity_rows.append((q["id"], dialect, em, cf))

identity_df = pd.DataFrame(identity_rows, columns=["query_id", "dialect", "em", "component_f1"])
n_em_fail = (~identity_df["em"]).sum()
n_cf_fail = (identity_df["component_f1"] < 0.999).sum()
print(f"Identity test: {len(identity_df)} cases")
print(f"  EM failures: {n_em_fail}")
print(f"  Component F1 < 1.0: {n_cf_fail}")
if n_em_fail or n_cf_fail:
    display(identity_df[(~identity_df["em"]) | (identity_df["component_f1"] < 0.999)])
    raise AssertionError("Identity sanity test failed: canonicalisation has a bug, do not trust metrics.")
print("PASS")

## Compute structural metrics on the real records

Only records that produced a non-null `generated_query` get metric values; failed translations get NaN so the report can distinguish 'failed to translate at all' from 'translated but wrong'.

In [ ]:
records = json.loads(RECORDS_PATH.read_text())
rows = []
for r in records:
    translated = r.get("generated_query")
    expected = r["expected_query"]
    dialect = r["dialect"]
    if not translated:
        rows.append({
            "query_id": r["query_id"], "schema_name": r["schema_name"], "dialect": dialect,
            "difficulty": r["difficulty"], "exact_match": False, "component_f1_overall": 0.0,
            "f1_node_labels": 0.0, "f1_edge_types": 0.0, "f1_directions": 0.0,
            "f1_where": 0.0, "f1_return": 0.0, "f1_order": 0.0, "f1_limit": 0.0, "f1_aggregations": 0.0,
        })
        continue
    em = exact_match(translated, expected, dialect)
    cf = component_f1(translated, expected, dialect)
    rows.append({
        "query_id": r["query_id"], "schema_name": r["schema_name"], "dialect": dialect,
        "difficulty": r["difficulty"], "exact_match": em, "component_f1_overall": cf["overall"],
        "f1_node_labels": cf["node_labels"], "f1_edge_types": cf["edge_types"],
        "f1_directions": cf["directions"], "f1_where": cf["where"], "f1_return": cf["return"],
        "f1_order": cf["order"], "f1_limit": cf["limit"], "f1_aggregations": cf["aggregations"],
    })
struct_df = pd.DataFrame(rows)
struct_df.head(8)

## Overall + stratified summaries

In [ ]:
metric_cols = [
    "exact_match", "component_f1_overall",
    "f1_node_labels", "f1_edge_types", "f1_directions",
    "f1_where", "f1_return", "f1_order", "f1_limit", "f1_aggregations",
]
print("Overall:")
display(struct_df[metric_cols].mean().to_frame("mean"))
print("\nBy schema × dialect:")
display(struct_df.groupby(["schema_name", "dialect"])[metric_cols].mean())
print("\nBy difficulty:")
display(struct_df.groupby("difficulty")[metric_cols].mean().reindex(["easy", "medium", "hard"]))

In [ ]:
struct_df.to_csv(OUT_CSV, index=False)
print(f"Wrote {len(struct_df)} rows to {OUT_CSV}")